In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import sem, t

# --- FILE 1: Table1_Complete_150_Mutations.xlsx ---
try:
    df_table1 = pd.read_excel('/content/Table1_Complete_150_Mutations.xlsx')
    print("\n--- Loaded Table1_Complete_150_Mutations.xlsx ---")
    display(df_table1.head())
except FileNotFoundError:
    print("Error: Table1_Complete_150_Mutations.xlsx not found.")
    df_table1 = pd.DataFrame() # Create empty DataFrame to avoid errors later

# --- FILE 2: Real_RMSD_Values_new_pdb.xlsx ---
try:
    # Corrected filename as per user's provided context
    df_rmsd = pd.read_excel('/content/Supplementary Table S3 - Real_RMSD_Values_new_pdb.xlsx')
    print("\n--- Loaded Real_RMSD_Values_new_pdb.xlsx ---")
    display(df_rmsd.head())
except FileNotFoundError:
    print("Error: Real_RMSD_Values_new_pdb.xlsx not found at specified path.")
    df_rmsd = pd.DataFrame() # Create empty DataFrame to avoid errors later

# --- FILE 3: tp53_mutation_frequencies.csv ---
try:
    df_frequencies = pd.read_csv('/content/tp53_mutation_frequencies.csv')
    print("\n--- Loaded tp53_mutation_frequencies.csv ---")
    display(df_frequencies.head())
except FileNotFoundError:
    print("Error: tp53_mutation_frequencies.csv not found.")
    df_frequencies = pd.DataFrame() # Create empty DataFrame to avoid errors later

# --- FILE 4: ESM_Atlas_TP53_Clusters_Summary.csv ---
try:
    df_esm_atlas = pd.read_csv('/content/ESM_Atlas_TP53_Clusters_Summary.csv')
    print("\n--- Loaded ESM_Atlas_TP53_Clusters_Summary.csv ---")
    display(df_esm_atlas.head())
except FileNotFoundError:
    print("Error: ESM_Atlas_TP53_Clusters_Summary.csv not found.")
    df_esm_atlas = pd.DataFrame() # Create empty DataFrame to avoid errors later

print("\nData loading complete. Please review the displayed heads to ensure correct loading.")



--- Loaded Table1_Complete_150_Mutations.xlsx ---


,Mutation,COSMIC ID,COSMIC Count,COSMIC %,Affinity (kcal/mol),Rank,Mean pLDDT,Min pLDDT,Max pLDDT,DBD Residues,Has DBD,Has Tetramerization,Structure_Confidence_Category
0,TP53_K132N,NaN,371,0.15,-8.80,1,88.102659,51.46,98.97,102-292,Yes,Yes,High
1,TP53_M198I,NaN,575,0.24,-8.75,2,87.353283,60.18,98.93,102-292,Yes,Yes,High
2,TP53_C199Y,NaN,410,0.17,-8.73,3,87.452889,56.12,98.94,102-292,Yes,Yes,High
3,TP53_R119H,NaN,325,0.13,-8.67,4,87.712908,52.44,98.97,102-292,Yes,Yes,High
4,TP53_Y88C,NaN,843,0.35,-8.57,5,87.727771,60.17,98.94,102-292,Yes,Yes,High



--- Loaded Real_RMSD_Values_new_pdb.xlsx ---


,Mutation,RMSD (Å)
0,TP53_P190L,1.131762
1,TP53_H154R,0.925248
2,TP53_R158L,0.933395
3,TP53_R282W,1.075952
4,TP53_R123W,0.997108



--- Loaded tp53_mutation_frequencies.csv ---


,Mutation,Count,Percentage
0,p.?,23687,9.74
1,p.R175H,5866,2.41
2,p.R136H,4190,1.72
3,p.R248Q,3717,1.53
4,p.R273C,2886,1.19



--- Loaded ESM_Atlas_TP53_Clusters_Summary.csv ---


,cluster_name,num_proteins,pfam_domains,cluster_id,mean_domain_coverage,pct_characterized,taxonomy_name,taxonomy_rank
0,08fce4fbede299ab48f88e752fbecd4a,50,P53 DNA-binding domain (44); SAM domain (Steri...,08fce4fbede299ab48f88e752fbecd4a,0.449957,88,Opisthokonta,clade
1,cbd2d143249c8a7ae0f56ac1a24cc12f,95,"CEP-1, DNA binding (1); P53 DNA-binding domain...",cbd2d143249c8a7ae0f56ac1a24cc12f,0.576741,97,Eumetazoa,clade
2,1475a16d26c7ee98cff0ee07d712039e,64,P53 transactivation motif (2); P53 DNA-binding...,1475a16d26c7ee98cff0ee07d712039e,0.493378,92,Opisthokonta,clade
3,c6fc1db36bacbcd28570023aaed7d226,98,P53 tetramerisation motif (20); SAM domain (St...,c6fc1db36bacbcd28570023aaed7d226,0.548482,97,Eumetazoa,clade
4,4cebba487eb6eced3d5a5e65408881bb,90,P53 transactivation motif (21); P53 DNA-bindin...,4cebba487eb6eced3d5a5e65408881bb,0.525130,99,Eumetazoa,clade



Data loading complete. Please review the displayed heads to ensure correct loading.


In [8]:
# Merge df_table1 and df_rmsd for calculations involving both structural and affinity data
if not df_table1.empty and not df_rmsd.empty:
    merged_df = pd.merge(df_table1, df_rmsd, on='Mutation', how='left')
    print("\n--- Merged df_table1 and df_rmsd ---")
    display(merged_df.head())
else:
    print("Cannot merge dataframes because one or more are empty.")
    merged_df = pd.DataFrame()


# --- CALCULATION 1: Therapeutic Prioritization Scores ---
print("\n--- Performing CALCULATION 1: Therapeutic Prioritization Scores ---")

if not merged_df.empty:
    # Ensure RMSD is numeric and handle potential NaNs (though instructions imply all 150 will have values)
    merged_df['RMSD (Å)'] = pd.to_numeric(merged_df['RMSD (Å)'], errors='coerce')
    merged_df = merged_df.dropna(subset=['RMSD (Å)', 'Affinity (kcal/mol)']) # Drop rows where essential values are missing

    # structural_score = 1 / (1 + RMSD)
    merged_df['structural_score'] = 1 / (1 + merged_df['RMSD (Å)'])

    # affinity_score = (Affinity - min_affinity) / (max_affinity - min_affinity)
    min_affinity = merged_df['Affinity (kcal/mol)'].min()
    max_affinity = merged_df['Affinity (kcal/mol)'].max()
    # Handle case where min_affinity == max_affinity to avoid division by zero
    if max_affinity == min_affinity:
        merged_df['affinity_score'] = 0.5 # Assign a neutral score if no variation
    else:
        merged_df['affinity_score'] = (merged_df['Affinity (kcal/mol)'] - min_affinity) / (max_affinity - min_affinity)

    # functional_score based on mutation class
    # Standardize mutation names for mapping
    merged_df['Mutation_Std'] = merged_df['Mutation'].str.replace('TP53_', '', regex=False)

    mutation_functional_scores = {
        'Y220C': 1.0,
        'R175H': 0.8, 'R282W': 0.8,
        'R248Q': 0.7, 'R273H': 0.7, 'R273C': 0.7,
        'V157F': 0.6, 'R158L': 0.6,
        'A138V': 0.4, 'R181C': 0.4
    }
    merged_df['functional_score'] = merged_df['Mutation_Std'].map(mutation_functional_scores).fillna(0.5)

    # clinical_anchor
    merged_df['clinical_anchor'] = merged_df['Mutation_Std'].apply(lambda x: 1.0 if x == 'Y220C' else 0.0)

    # therapeutic_score = (0.4 * structural_score) + (0.3 * affinity_score) + (0.2 * functional_score) + (0.1 * clinical_anchor)
    merged_df['therapeutic_score'] = (
        0.4 * merged_df['structural_score'] +
        0.3 * merged_df['affinity_score'] +
        0.2 * merged_df['functional_score'] +
        0.1 * merged_df['clinical_anchor']
    )

    # Select and reorder columns for output
    therapeutic_scores_df = merged_df[[
        'Mutation', 'RMSD (Å)', 'Affinity (kcal/mol)', 'functional_score',
        'structural_score', 'affinity_score', 'clinical_anchor', 'therapeutic_score', 'Rank'
    ]].copy()

    # Rename columns for clarity in output
    therapeutic_scores_df = therapeutic_scores_df.rename(columns={
        'RMSD (Å)': 'RMSD',
        'Affinity (kcal/mol)': 'Affinity'
    })

    # Save to CSV
    output_file_calc1 = 'therapeutic_prioritization_scores.csv'
    therapeutic_scores_df.to_csv(output_file_calc1, index=False)
    print(f"Saved therapeutic prioritization scores to {output_file_calc1}")
    display(therapeutic_scores_df.head())
else:
    print("CALCULATION 1: NOT CALCULATED - Merged dataframe is empty due to missing input files.")
    therapeutic_scores_df = pd.DataFrame()

# --- CALCULATION 2: Top 10 Highest Therapeutic Scores ---
print("\n--- Performing CALCULATION 2: Top 10 Highest Therapeutic Scores ---")

if not therapeutic_scores_df.empty:
    top10_therapeutic_scores_df = therapeutic_scores_df.sort_values(
        by='therapeutic_score', ascending=False
    ).head(10).reset_index(drop=True)

    # Select and reorder columns for output
    top10_output_df = top10_therapeutic_scores_df[[
        'Rank', 'Mutation', 'therapeutic_score', 'RMSD', 'Affinity', 'functional_score'
    ]]

    # Save to CSV
    output_file_calc2 = 'top10_therapeutic_scores.csv'
    top10_output_df.to_csv(output_file_calc2, index=False)
    print(f"Saved top 10 therapeutic scores to {output_file_calc2}")
    display(top10_output_df)
else:
    print("CALCULATION 2: NOT CALCULATED - Therapeutic scores dataframe is empty.")

# --- CALCULATION 3: ML Performance Metrics (Table 2) ---
print("\n--- Performing CALCULATION 3: ML Performance Metrics ---")

if not merged_df.empty and not df_frequencies.empty:
    # Prepare data for ML
    ml_df = merged_df.copy()

    # 1. Create pathogenicity label column
    # Standardize mutation names for merging with frequencies
    ml_df['Mutation_freq_merge'] = ml_df['Mutation'].str.replace('TP53_', 'p.', regex=False)
    df_frequencies['Mutation_freq_merge'] = df_frequencies['Mutation']

    ml_df = pd.merge(ml_df, df_frequencies[['Mutation_freq_merge', 'Count']],
                     on='Mutation_freq_merge', how='left', suffixes=('_table1', '_freq'))

    # Use COSMIC Count from Table1 first, then fallback to Frequencies Count if available, otherwise 0
    ml_df['COSMIC_Count_final'] = ml_df['COSMIC Count'].fillna(ml_df['Count']).fillna(0)

    # Pathogenicity logic
    # Known cancer mutations from the prompt that might not have COSMIC Count > 1000
    known_cancer_mutations = {
        'R175H', 'R282W', 'R248Q', 'R273H', 'R273C',
        'V157F', 'R158L', 'A138V', 'R181C', 'Y220C' # Y220C is a known target, implying pathogenicity
    }
    ml_df['Mutation_Std'] = ml_df['Mutation'].str.replace('TP53_', '', regex=False)

    def get_pathogenicity(row):
        if row['COSMIC_Count_final'] > 1000:
            return 1 # Pathogenic
        elif row['COSMIC_Count_final'] < 100 and row['Mutation_Std'] in known_cancer_mutations:
            return 1 # Pathogenic
        else:
            return 0 # Benign

    ml_df['pathogenicity'] = ml_df.apply(get_pathogenicity, axis=1)

    # Convert 'Has Tetramerization' to 1/0
    ml_df['Has Tetramerization_numeric'] = ml_df['Has Tetramerization'].map({'Yes': 1, 'No': 0})
    # Fill NaN for tetramerization if any, perhaps with the most common value or 0
    ml_df['Has Tetramerization_numeric'] = ml_df['Has Tetramerization_numeric'].fillna(0) # Q331* and Q292* have 'No'

    # Features to use
    features = [
        'Affinity (kcal/mol)',
        'RMSD (Å)',
        'Mean pLDDT',
        'COSMIC_Count_final',
        'Has Tetramerization_numeric',
        'structural_score' # from Calculation 1
    ]

    # Impute missing values in features *before* splitting for cross-validation
    # If 'structural_score' is missing (due to missing RMSD), fill it as well
    if 'structural_score' not in ml_df.columns:
        print("Warning: 'structural_score' not found in ml_df. It is required for ML features.")
        # This should not happen if Calc 1 ran successfully, but as a safeguard.
        ml_df['structural_score'] = 0.5 # Neutral value if it somehow went missing

    # Ensure all feature columns exist and are numeric
    for feature in features:
        if feature not in ml_df.columns:
            print(f"Warning: Feature '{feature}' not found in ml_df. Setting to 0.")
            ml_df[feature] = 0.0
        ml_df[feature] = pd.to_numeric(ml_df[feature], errors='coerce')

    X = ml_df[features]
    y = ml_df['pathogenicity']

    # Impute NaNs in X using the mean of each column from the *entire* X before CV splits
    # This is a simple imputation strategy, more complex strategies might use pipelines
    for feature in features:
        if X[feature].isnull().any():
            mean_val = X[feature].mean()
            X[feature] = X[feature].fillna(mean_val)

    # Drop rows with any remaining NaN values in features or target, or handle imputation before this step
    initial_rows = ml_df.shape[0]
    ml_df_cleaned = ml_df.dropna(subset=features + ['pathogenicity'])
    if ml_df_cleaned.shape[0] < initial_rows:
        print(f"Warning: Dropped {initial_rows - ml_df_cleaned.shape[0]} rows due to missing values in ML features/target.")
    X = ml_df_cleaned[features]
    y = ml_df_cleaned['pathogenicity']

    if X.empty or y.empty or len(y.unique()) < 2: # Check if there's enough data and at least two classes
        print("CALCULATION 3: NOT CALCULATED - Insufficient data or only one class for ML model training after cleaning.")
        ml_performance_metrics_df = pd.DataFrame()
    else:
        # Bootstrap function for CI
        def bootstrap_metric(y_true, y_pred, metric_func, n_bootstraps=1000, alpha=0.05):
            n_size = len(y_true)
            bootstrapped_metrics = []
            for _ in range(n_bootstraps):
                indices = np.random.choice(n_size, n_size, replace=True)
                if len(np.unique(y_true[indices])) < 2: # Ensure at least two classes in bootstrap sample
                    continue
                bootstrapped_metrics.append(metric_func(y_true[indices], y_pred[indices]))
            if not bootstrapped_metrics:
                return np.nan, (np.nan, np.nan)
            mean_metric = np.mean(bootstrapped_metrics)
            lower = np.percentile(bootstrapped_metrics, (alpha / 2) * 100)
            upper = np.percentile(bootstrapped_metrics, (1 - alpha / 2) * 100)
            return mean_metric, (lower, upper)

        # Models
        models = {
            'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'),
            'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
            'XGBoost': xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', n_estimators=100)
        }

        results = []
        n_splits = 5
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

        for name, model in models.items():
            roc_aucs, accuracies, precisions, recalls, f1s = [], [], [], [], []
            all_y_test, all_y_pred, all_y_proba = np.array([]), np.array([]), np.array([])

            for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
                X_train, X_test = X.iloc[train_index], X.iloc[test_index]
                y_train, y_test = y.iloc[train_index], y.iloc[test_index]

                # Simple imputation for training and testing sets based on training data
                for feature in features:
                    if X_train[feature].isnull().any():
                        mean_val = X_train[feature].mean()
                        X_train[feature] = X_train[feature].fillna(mean_val)
                        X_test[feature] = X_test[feature].fillna(mean_val)

                if len(y_train.unique()) < 2 or len(y_test.unique()) < 2: # Skip fold if insufficient classes
                    print(f"Warning: Skipping fold {fold} for {name} due to insufficient classes in train/test set.")
                    continue

                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                y_proba = model.predict_proba(X_test)[:, 1]

                roc_aucs.append(roc_auc_score(y_test, y_proba))
                accuracies.append(accuracy_score(y_test, y_pred))
                precisions.append(precision_score(y_test, y_pred, zero_division=0))
                recalls.append(recall_score(y_test, y_pred, zero_division=0))
                f1s.append(f1_score(y_test, y_pred, zero_division=0))

                all_y_test = np.concatenate((all_y_test, y_test))
                all_y_pred = np.concatenate((all_y_pred, y_pred))
                all_y_proba = np.concatenate((all_y_proba, y_proba))

            # Calculate mean and CI for AUROC and Accuracy
            mean_auroc, ci_auroc = bootstrap_metric(all_y_test, all_y_proba, roc_auc_score)
            mean_accuracy, ci_accuracy = bootstrap_metric(all_y_test, all_y_pred, accuracy_score)

            # Calculate ECE for calibration (requires probabilites)
            # This is a simplified ECE calculation. For a full implementation, usually involves binning.
            # For consistency with problem statement, a placeholder is used here if a specific ECE library isn't available.
            calibration_ece = "NOT CALCULATED (Specific ECE calculation not implemented)"
            try:
                from sklearn.calibration import calibration_curve
                from sklearn.metrics import brier_score_loss
                # Example ECE (Expected Calibration Error) could be approximated or calculated using dedicated libraries.
                # For now, using a placeholder as it's not a standard scikit-learn metric with easy CI.
                # If a robust ECE is needed, a specific implementation or library (e.g., 'netcal') would be required.
                calibration_ece = np.nan # Placeholder for now, can be implemented with more detail if required.
            except ImportError:
                pass

            results.append({
                'Method': name,
                'Accuracy': f"{mean_accuracy:.3f} ({ci_accuracy[0]:.3f}-{ci_accuracy[1]:.3f})",
                'AUROC': f"{mean_auroc:.3f} ({ci_auroc[0]:.3f}-{ci_auroc[1]:.3f})",
                'Precision': f"{np.mean(precisions):.3f}",
                'Recall': f"{np.mean(recalls):.3f}",
                'F1_Score': f"{np.mean(f1s):.3f}",
                'Calibration_ECE': calibration_ece
            })

        # Add published benchmark values
        benchmarks = [
            {'Method': 'SIFT', 'Accuracy': 'NOT CALCULATED', 'AUROC': '0.78', 'Precision': 'NOT CALCULATED', 'Recall': 'NOT CALCULATED', 'F1_Score': 'NOT CALCULATED', 'Calibration_ECE': 'NOT CALCULATED'},
            {'Method': 'PolyPhen-2', 'Accuracy': 'NOT CALCULATED', 'AUROC': '0.81', 'Precision': 'NOT CALCULATED', 'Recall': 'NOT CALCULATED', 'F1_Score': 'NOT CALCULATED', 'Calibration_ECE': 'NOT CALCULATED'},
            {'Method': 'AlphaMissense', 'Accuracy': 'NOT CALCULATED', 'AUROC': '0.88', 'Precision': 'NOT CALCULATED', 'Recall': 'NOT CALCULATED', 'F1_Score': 'NOT CALCULATED', 'Calibration_ECE': 'NOT CALCULATED'}
        ]
        ml_performance_metrics_df = pd.DataFrame(results + benchmarks)

        # Save to CSV
        output_file_calc3 = 'ml_performance_metrics.csv'
        ml_performance_metrics_df.to_csv(output_file_calc3, index=False)
        print(f"Saved ML performance metrics to {output_file_calc3}")
        display(ml_performance_metrics_df)
else:
    print("CALCULATION 3: NOT CALCULATED - Merged dataframe or frequencies dataframe is empty.")
    ml_performance_metrics_df = pd.DataFrame()


# --- CALCULATION 4: Compare Quantum vs Classical Docking (Figure 5) ---
print("\n--- Performing CALCULATION 4: Compare Quantum vs Classical Docking ---")

# Given times
esmfold2_time_hours = 6 + 17/60 + 45/3600 # 6.30 hours
autodock_vina_time_hours = 10 + 39/60 + 19/3600 # 10.65 hours
quantum_docking_time_hours = 2.0
ml_training_time_hours = 2.0
hyperparameter_optimization_time_hours = 1.0

# Calculate Speedup
speedup = autodock_vina_time_hours / quantum_docking_time_hours

# Calculate Total pipeline time
total_pipeline_time = (
    esmfold2_time_hours +
    autodock_vina_time_hours + # This is for generating structures for docking, not the docking itself, but included in pipeline
    quantum_docking_time_hours +
    ml_training_time_hours +
    hyperparameter_optimization_time_hours
)

# Create DataFrame
computational_timing_data = [
    {'Step': 'ESMFold2', 'Time_hours': esmfold2_time_hours, 'Time_readable': '6 hours, 17 minutes, 45 seconds', 'Source': 'Given'},
    {'Step': 'AutoDock Vina', 'Time_hours': autodock_vina_time_hours, 'Time_readable': '10 hours, 39 minutes, 19 seconds', 'Source': 'Given'},
    {'Step': 'Quantum Docking', 'Time_hours': quantum_docking_time_hours, 'Time_readable': '2.0 hours', 'Source': 'Estimated'},
    {'Step': 'ML Training', 'Time_hours': ml_training_time_hours, 'Time_readable': '2.0 hours', 'Source': 'Estimated'},
    {'Step': 'Hyperparameter Optimization', 'Time_hours': hyperparameter_optimization_time_hours, 'Time_readable': '1.0 hour', 'Source': 'Estimated'},
    {'Step': 'Speedup (AutoDock Vina / Quantum Docking)', 'Time_hours': speedup, 'Time_readable': f'{speedup:.2f}x', 'Source': 'Calculated'},
    {'Step': 'Total Pipeline Time', 'Time_hours': total_pipeline_time, 'Time_readable': f'{total_pipeline_time:.2f} hours', 'Source': 'Calculated'}
]
computational_timing_df = pd.DataFrame(computational_timing_data)

# Save to CSV
output_file_calc4 = 'computational_timing.csv'
computational_timing_df.to_csv(output_file_calc4, index=False)
print(f"Saved computational timing to {output_file_calc4}")
display(computational_timing_df)

# --- CALCULATION 5: Docking Statistics Summary ---
print("\n--- Performing CALCULATION 5: Docking Statistics Summary ---")

if not df_table1.empty:
    affinity_data = df_table1['Affinity (kcal/mol)']

    num_mutations_docked = len(affinity_data) # Assuming df_table1 contains all 150
    successfully_docked = num_mutations_docked # As per problem statement, should be 150
    mean_affinity = affinity_data.mean()
    median_affinity = affinity_data.median()
    best_affinity_val = affinity_data.min()
    best_affinity_mutation = df_table1.loc[affinity_data.idxmin(), 'Mutation']
    worst_affinity_val = affinity_data.max()
    worst_affinity_mutation = df_table1.loc[affinity_data.idxmax(), 'Mutation']
    std_dev_affinity = affinity_data.std()

    docking_stats = [
        {'Metric': 'Number of mutations docked', 'Value': num_mutations_docked},
        {'Metric': 'Successfully docked', 'Value': successfully_docked},
        {'Metric': 'Mean affinity', 'Value': f'{mean_affinity:.2f}'},
        {'Metric': 'Median affinity', 'Value': f'{median_affinity:.2f}'},
        {'Metric': 'Best affinity', 'Value': f'{best_affinity_val:.2f}'},
        {'Metric': 'Best affinity mutation', 'Value': best_affinity_mutation},
        {'Metric': 'Worst affinity', 'Value': f'{worst_affinity_val:.2f}'},
        {'Metric': 'Worst affinity mutation', 'Value': worst_affinity_mutation},
        {'Metric': 'Standard deviation', 'Value': f'{std_dev_affinity:.2f}'}
    ]
    docking_statistics_summary_df = pd.DataFrame(docking_stats)

    # Save to CSV
    output_file_calc5 = 'docking_statistics_summary.csv'
    docking_statistics_summary_df.to_csv(output_file_calc5, index=False)
    print(f"Saved docking statistics summary to {output_file_calc5}")
    display(docking_statistics_summary_df)
else:
    print("CALCULATION 5: NOT CALCULATED - Table1_Complete_150_Mutations.xlsx not loaded or is empty.")

# --- CALCULATION 6: ESM Atlas Summary ---
print("\n--- Performing CALCULATION 6: ESM Atlas Summary ---")

if not df_esm_atlas.empty:
    total_proteins_across_clusters = df_esm_atlas['num_proteins'].sum()
    num_clusters = df_esm_atlas.shape[0] # Should be 10
    mean_domain_coverage_per_cluster = df_esm_atlas['mean_domain_coverage'].mean()

    # For taxonomy distribution, aggregate counts for each taxonomy_name
    taxonomy_distribution = df_esm_atlas.groupby('taxonomy_name')['num_proteins'].sum().reset_index()
    taxonomy_distribution.rename(columns={'num_proteins': 'Total_Proteins_in_Taxonomy'}, inplace=True)

    # Prepare output, grouping by cluster for the main summary, and then taxonomy separately
    # The request asks for 'Cluster_ID, Proteins, Mean_Coverage, Taxonomy' for the output file
    # This implies a per-cluster summary including its taxonomy info
    esm_atlas_summary_df = df_esm_atlas[[
        'cluster_id', 'num_proteins', 'mean_domain_coverage', 'taxonomy_name'
    ]].copy()
    esm_atlas_summary_df.rename(columns={
        'cluster_id': 'Cluster_ID',
        'num_proteins': 'Proteins',
        'mean_domain_coverage': 'Mean_Coverage',
        'taxonomy_name': 'Taxonomy'
    }, inplace=True)

    # Save to CSV
    output_file_calc6 = 'esm_atlas_summary.csv'
    esm_atlas_summary_df.to_csv(output_file_calc6, index=False)
    print(f"Saved ESM Atlas summary to {output_file_calc6}")
    display(esm_atlas_summary_df)

    print(f"\nTotal proteins across all clusters: {total_proteins_across_clusters}")
    print(f"Number of clusters: {num_clusters}")
    print(f"Mean domain coverage across clusters: {mean_domain_coverage_per_cluster:.2f}")
    print("Taxonomy Distribution (Total Proteins):")
    display(taxonomy_distribution)
else:
    print("CALCULATION 6: NOT CALCULATED - ESM_Atlas_TP53_Clusters_Summary.csv not loaded or is empty.")

# --- CALCULATION 7: Structure Confidence Summary ---
print("\n--- Performing CALCULATION 7: Structure Confidence Summary ---")

if not df_table1.empty:
    # Total proteins analyzed (150)
    total_proteins_analyzed = df_table1.shape[0]

    # pLDDT data
    mean_plddt = df_table1['Mean pLDDT'].mean()
    median_plddt = df_table1['Mean pLDDT'].median()
    min_plddt = df_table1['Min pLDDT'].min()
    max_plddt = df_table1['Max pLDDT'].max()

    # Number with DBD (150)
    num_with_dbd = df_table1[df_table1['Has DBD'] == 'Yes'].shape[0]

    # Number with Tetramerization (148)
    num_with_tetramerization = df_table1[df_table1['Has Tetramerization'] == 'Yes'].shape[0]

    # Percentage with Tetramerization (98.67%)
    percentage_with_tetramerization = (num_with_tetramerization / total_proteins_analyzed) * 100

    structure_confidence_stats = [
        {'Metric': 'Total proteins analyzed', 'Value': total_proteins_analyzed},
        {'Metric': 'Mean pLDDT', 'Value': f'{mean_plddt:.2f}'},
        {'Metric': 'Median pLDDT', 'Value': f'{median_plddt:.2f}'},
        {'Metric': 'Min pLDDT', 'Value': f'{min_plddt:.2f}'},
        {'Metric': 'Max pLDDT', 'Value': f'{max_plddt:.2f}'},
        {'Metric': 'Number with DBD', 'Value': num_with_dbd},
        {'Metric': 'Number with Tetramerization', 'Value': num_with_tetramerization},
        {'Metric': 'Percentage with Tetramerization', 'Value': f'{percentage_with_tetramerization:.2f}%'}
    ]
    structure_confidence_summary_df = pd.DataFrame(structure_confidence_stats)

    # Save to CSV
    output_file_calc7 = 'structure_confidence_summary.csv'
    structure_confidence_summary_df.to_csv(output_file_calc7, index=False)
    print(f"Saved structure confidence summary to {output_file_calc7}")
    display(structure_confidence_summary_df)
else:
    print("CALCULATION 7: NOT CALCULATED - Table1_Complete_150_Mutations.xlsx not loaded or is empty.")

print("\n--- All calculations complete. ---")



--- Merged df_table1 and df_rmsd ---


,Mutation,COSMIC ID,COSMIC Count,COSMIC %,Affinity (kcal/mol),Rank,Mean pLDDT,Min pLDDT,Max pLDDT,DBD Residues,Has DBD,Has Tetramerization,Structure_Confidence_Category,RMSD (Å)
0,TP53_K132N,NaN,371,0.15,-8.80,1,88.102659,51.46,98.97,102-292,Yes,Yes,High,0.887400
1,TP53_M198I,NaN,575,0.24,-8.75,2,87.353283,60.18,98.93,102-292,Yes,Yes,High,1.061334
2,TP53_C199Y,NaN,410,0.17,-8.73,3,87.452889,56.12,98.94,102-292,Yes,Yes,High,1.059662
3,TP53_R119H,NaN,325,0.13,-8.67,4,87.712908,52.44,98.97,102-292,Yes,Yes,High,1.149155
4,TP53_Y88C,NaN,843,0.35,-8.57,5,87.727771,60.17,98.94,102-292,Yes,Yes,High,1.042674



--- Performing CALCULATION 1: Therapeutic Prioritization Scores ---
Saved therapeutic prioritization scores to therapeutic_prioritization_scores.csv


,Mutation,RMSD,Affinity,functional_score,structural_score,affinity_score,clinical_anchor,therapeutic_score,Rank
0,TP53_K132N,0.887400,-8.80,0.5,0.529829,0.000000,0.0,0.311932,1
1,TP53_M198I,1.061334,-8.75,0.5,0.485123,0.020492,0.0,0.300197,2
2,TP53_C199Y,1.059662,-8.73,0.5,0.485517,0.028689,0.0,0.302813,3
3,TP53_R119H,1.149155,-8.67,0.5,0.465299,0.053279,0.0,0.302103,4
4,TP53_Y88C,1.042674,-8.57,0.5,0.489554,0.094262,0.0,0.324100,5



--- Performing CALCULATION 2: Top 10 Highest Therapeutic Scores ---
Saved top 10 therapeutic scores to top10_therapeutic_scores.csv


...


Total proteins across all clusters: 725
Number of clusters: 10
Mean domain coverage across clusters: 0.55
Taxonomy Distribution (Total Proteins):


,taxonomy_name,Total_Proteins_in_Taxonomy
0,Eumetazoa,406
1,Opisthokonta,183
2,root,136



--- Performing CALCULATION 7: Structure Confidence Summary ---
Saved structure confidence summary to structure_confidence_summary.csv


,Metric,Value
0,Total proteins analyzed,150
1,Mean pLDDT,87.55
2,Median pLDDT,87.52
3,Min pLDDT,34.10
4,Max pLDDT,98.97
5,Number with DBD,150
6,Number with Tetramerization,148
7,Percentage with Tetramerization,98.67%



--- All calculations complete. ---


In [10]:
print("ML Performance Metrics:")
display(ml_performance_metrics_df)

ML Performance Metrics:


,Method,Accuracy,AUROC,Precision,Recall,F1_Score,Calibration_ECE
0,Logistic Regression,0.993 (0.980-1.000),1.000 (1.000-1.000),1.000,0.967,0.982,NaN
1,Random Forest,0.993 (0.980-1.000),0.999 (0.996-1.000),1.000,0.967,0.982,NaN
2,XGBoost,0.993 (0.980-1.000),0.973 (0.907-1.000),1.000,0.967,0.982,NaN
3,SIFT,NOT CALCULATED,0.78,NOT CALCULATED,NOT CALCULATED,NOT CALCULATED,NOT CALCULATED
4,PolyPhen-2,NOT CALCULATED,0.81,NOT CALCULATED,NOT CALCULATED,NOT CALCULATED,NOT CALCULATED
5,AlphaMissense,NOT CALCULATED,0.88,NOT CALCULATED,NOT CALCULATED,NOT CALCULATED,NOT CALCULATED


In [11]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=ml_performance_metrics_df)

https://docs.google.com/spreadsheets/d/1_QTsI8YWXo_jox-RSM_EVqw-2fBpcvSRnmPkr4c_xdg/edit#gid=0


In [12]:
from google.colab import files

zip_file_name = 'tp53_analysis_results.zip'
if os.path.exists(zip_file_name):
    print(f"Re-downloading '{zip_file_name}'...")
    files.download(zip_file_name)
else:
    print(f"Error: '{zip_file_name}' not found. Please ensure the previous zipping step was successful.")

Re-downloading 'tp53_analysis_results.zip'...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

This table summarizes the performance of the trained Machine Learning models (Logistic Regression, Random Forest, and XGBoost) in predicting TP53 mutation pathogenicity. It also includes published benchmark values for comparison. The metrics include Accuracy, AUROC (Area Under the Receiver Operating Characteristic curve), Precision, Recall, and F1-Score, along with 95% Confidence Intervals for Accuracy and AUROC, calculated using bootstrapping.

In [9]:
import os
import zipfile
from google.colab import files

# Define the output directory and zip file name
output_dir_name = 'tp53_analysis_results'
zip_file_name = f'{output_dir_name}.zip'

# Create the output directory if it doesn't exist
if not os.path.exists(output_dir_name):
    os.makedirs(output_dir_name)

# List of generated CSV files
csv_files = [
    'therapeutic_prioritization_scores.csv',
    'top10_therapeutic_scores.csv',
    'ml_performance_metrics.csv',
    'computational_timing.csv',
    'docking_statistics_summary.csv',
    'esm_atlas_summary.csv',
    'structure_confidence_summary.csv'
]

# Move each CSV file into the new directory
print(f"\nMoving CSV files to '{output_dir_name}/'...")
for f in csv_files:
    if os.path.exists(f):
        os.rename(f, os.path.join(output_dir_name, f))
        print(f"  - Moved {f}")
    else:
        print(f"  - Warning: {f} not found, skipping.")

# Zip the directory
print(f"\nZipping '{output_dir_name}/' into '{zip_file_name}'...")
with zipfile.ZipFile(zip_file_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(output_dir_name):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), output_dir_name))
print(f"Successfully created '{zip_file_name}'")

# Offer the zip file for download
print("\nYour results are ready for download:")
files.download(zip_file_name)



Moving CSV files to 'tp53_analysis_results/'...
  - Moved therapeutic_prioritization_scores.csv
  - Moved top10_therapeutic_scores.csv
  - Moved ml_performance_metrics.csv
  - Moved computational_timing.csv
  - Moved docking_statistics_summary.csv
  - Moved esm_atlas_summary.csv
  - Moved structure_confidence_summary.csv

Zipping 'tp53_analysis_results/' into 'tp53_analysis_results.zip'...
Successfully created 'tp53_analysis_results.zip'

Your results are ready for download:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>